[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TianshuangQiu/TorchCode/blob/master/templates/45_per_group_linear.ipynb)

# 🔴 Hard: Per-Group Linear (MoE Forward)

Given inputs routed to different "experts", apply the correct weight matrix to each input:

- `x` of shape `(N, d_in)` — batch of tokens
- `group_ids` of shape `(N,)` with values in `[0, G)` — expert assignment per token
- `W` of shape `(G, d_out, d_in)` — one weight matrix per expert
- `b` of shape `(G, d_out)` — one bias per expert

Return `y` of shape `(N, d_out)` where `y[i] = W[group_ids[i]] @ x[i] + b[group_ids[i]]`.

This is the core forward pass of **Mixture-of-Experts** when each token is routed to exactly one expert.

### Signature
```python
def per_group_linear(
    x: torch.Tensor,         # (N, d_in)
    group_ids: torch.Tensor, # (N,)
    W: torch.Tensor,         # (G, d_out, d_in)
    b: torch.Tensor,         # (G, d_out)
) -> torch.Tensor:           # (N, d_out)
    ...
```

### Rules
- Do **NOT** use Python `for` loops over `N` or `G`
- Gradients must flow correctly to `W` and `b`

### Example
```
x = [[1,0],[0,1],[1,1]]   group_ids = [0, 1, 0]
W[0] = identity, W[1] = 2*identity
b[0] = [0,0],    b[1] = [1,1]

output: [[1,0], [1,3], [1,1]]
```

> **Reduction step (say this before coding):** `y[i]` needs `W[group_ids[i]]` — this is **fancy indexing** into `W` along the first axis, giving shape `(N, d_out, d_in)`. Then it's a batched matrix-vector multiply.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def per_group_linear(
    x: torch.Tensor,
    group_ids: torch.Tensor,
    W: torch.Tensor,
    b: torch.Tensor,
) -> torch.Tensor:
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
x         = torch.tensor([[1.,0.],[0.,1.],[1.,1.]])
group_ids = torch.tensor([0, 1, 0])
W = torch.zeros(2, 2, 2)
W[0] = torch.eye(2)      # identity
W[1] = 2 * torch.eye(2)  # 2x identity
b = torch.zeros(2, 2)
b[1] = torch.tensor([1., 1.])

result = per_group_linear(x, group_ids, W, b)
print('output:', result.tolist())
print('expect: [[1.0, 0.0], [1.0, 3.0], [1.0, 1.0]]')

# Check gradient flow
W2 = W.clone().requires_grad_(True)
b2 = b.clone().requires_grad_(True)
per_group_linear(x, group_ids, W2, b2).sum().backward()
print('W.grad is not None:', W2.grad is not None)
print('b.grad is not None:', b2.grad is not None)

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("per_group_linear")